# OMBRIA QGSF v0.3 smoke gate

This one-seed/two-epoch run validates the quality-gated method, event evaluation, prespecified contrast export, and artifact packaging. Its scores are not scientific evidence.

Expected to be much shorter than Full; actual time is recorded.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

WORKING = Path('/kaggle/working')
REPO_URL = 'https://github.com/NewRudy/geoai-ombria-robustness.git'
SOURCE_REF = 'v0.3.1-quality-gated'
project = WORKING / 'geoai-ombria-robustness'
os.chdir(WORKING)
if project.exists():
    shutil.rmtree(project)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', SOURCE_REF, REPO_URL, str(project)], check=True)
os.chdir(project)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('locked source:', SOURCE_REF, commit)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.24', 'pillow>=10.0', 'matplotlib>=3.7'], check=True)
subprocess.run([sys.executable, 'scripts/ensure_cuda_compat.py'], check=True)
subprocess.run([sys.executable, 'scripts/check_cuda_runtime.py'], check=True)
scripts = [str(path) for path in Path('scripts').glob('*.py')]
subprocess.run([sys.executable, '-m', 'py_compile', *scripts], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


In [ ]:
env = os.environ.copy()
env.update({'MODE': 'smoke', 'PYTHON': sys.executable})
subprocess.run(['bash', 'scripts/run_quality_gated_v3_matrix.sh'], check=True, env=env)


In [ ]:
import hashlib, json
from zipfile import ZipFile
from IPython.display import FileLink, display

artifact = project / 'results' / 'ombria_quality_gated_v3_artifacts.zip'
with ZipFile(artifact) as archive:
    assert archive.testzip() is None
    names = archive.namelist()
    artifact_manifest_name = next(name for name in names if name.endswith('artifact_manifest.json'))
    artifact_manifest = json.loads(archive.read(artifact_manifest_name))
    for record in artifact_manifest['files']:
        assert record['path'] in names
        assert hashlib.sha256(archive.read(record['path'])).hexdigest() == record['sha256']
    checkpoint_manifest_name = next(name for name in names if name.endswith('checkpoint_manifest.json'))
    checkpoint_manifest = json.loads(archive.read(checkpoint_manifest_name))
    assert checkpoint_manifest['weights_included'] is True
    decision_manifest_name = next(name for name in names if name.endswith('decision_gate.json'))
    decision = json.loads(archive.read(decision_manifest_name))['decision']
assert decision["status"] == "pipeline_only"
print('artifact MB:', round(artifact.stat().st_size / 1024**2, 1))
display(FileLink(str(artifact)))
